### ЗАДАЧА: Распределение доставок между курьерами

Логистическая команда получает пакет заявок на доставки, которые нужно распределить между курьерами на электровелосипедах.
Нужно собрать систему, которая:
- принимает корректные доставки,
- отбрасывает неправильные или небезопасные заявки,
- уменьшает доступный заряд после успешно назначенного маршрута,
- ведёт отдельный журнал ошибок,
- помогает понять, какой курьер был загружен дольше всех и какому клиенту доставили самый большой вес.

In [ ]:
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict, Any

couriers = {
    'CR-1': {'zone': 'north', 'charge_min': 40, 'max_weight': 3.0},
    'CR-2': {'zone': 'south', 'charge_min': 30, 'max_weight': 2.0},
    'CR-3': {'zone': 'north', 'charge_min': 55, 'max_weight': 5.0},
}

# rows: delivery_id|courier_id|client|weight_kg|route_min
rows = [
    'DL-100|CR-1|Clinic|1.5|12',
    'DL-101|CR-2|Cafe|2.5|10',
    'DL-102|CR-9|Lab|1.0|8',
    'DL-103|CR-1|Shop|0|6',
    'DL-104|CR-3|Village|3.5|60',
    'DL-100|CR-3|Clinic|1.0|10',
    'DL-105|CR-3|School|2.0|20',
    'DL-106|CR-2|Pharmacy|1.0|15',
]

class DeliveryError(Exception):
    pass

class RowFormatError(DeliveryError):
    pass

class CourierNotFoundError(DeliveryError):
    pass

class WeightError(DeliveryError):
    pass

class RouteTimeError(DeliveryError):
    pass

class WeightLimitError(DeliveryError):
    pass

class ChargeReserveError(DeliveryError):
    pass

class DuplicateDeliveryError(DeliveryError):
    pass

@dataclass(order=True)
class Delivery:
    route_min: int
    delivery_id: str
    courier_id: str
    client: str
    weight_kg: float

class Courier:
    def __init__(self, courier_id, zone, charge_min, max_weight):
        # TODO: сохранить courier_id, zone, charge_min, max_weight
        self.courier_id = courier_id
        self.zone = zone
        self.charge_min = charge_min
        self.max_weight = max_weight
        # TODO: создать список deliveries
        self.deliveries = []

    def charge_left(self):
        # TODO: вернуть текущий остаток заряда в минутах
        return self.charge_min - self.total_route_time()

    def total_route_time(self):
        # TODO: вернуть сумму route_min по self.deliveries
        return sum(delivery.route_min for delivery in self.deliveries)

    def total_weight(self):
        # TODO: вернуть сумму weight_kg по self.deliveries
        return sum(delivery.weight_kg for delivery in self.deliveries)

    def assign(self, delivery):
        # TODO: если delivery.weight_kg > self.max_weight -> raise WeightLimitError(...)
        if delivery.weight_kg > self.max_weight:
            raise WeightLimitError(
                f"Вес {delivery.weight_kg} кг превышает максимальный допустимый вес {self.max_weight} кг для курьера {self.courier_id}"
            )
        # TODO: посчитать charge_after = charge_left() - delivery.route_min
        charge_after = self.charge_left() - delivery.route_min
        # TODO: если charge_after < 5 -> raise ChargeReserveError(...)
        if charge_after < 5:
            raise ChargeReserveError(
                f"менее 5 минут остаток заряда для курьера {self.courier_id}, текущий остаток заряда: {self.charge_left()} мин"
            )
        # TODO: добавить delivery в self.deliveries
        self.deliveries.append(delivery)
        # TODO: отсортировать self.deliveries
        self.deliveries.sort()

class CourierDispatchService:
    def __init__(self, couriers):
        # TODO: создать couriers вида courier_id -> Courier(...)
        self.couriers = {
            cid: Courier(cid, data['zone'], data['charge_min'], data['max_weight'])
            for cid, data in couriers.items()
        }
        # TODO: создать списки accepted и errors
        self.accepted: List[Delivery] = []
        self.errors: List[Tuple[str, str, str]] = []
        # TODO: создать множество processed_ids
        self.processed_ids: set = set()

    def parse_delivery(self, row):
        # TODO: split по '|'
        parts = row.split('|')
        # TODO: ожидать 5 частей: delivery_id, courier_id, client, weight_raw, route_raw
        if len(parts) != 5:
            raise RowFormatError(f"Неверный формат строки: ожидается 5 частей, получено {len(parts)}: {row}")
        delivery_id, courier_id, client, weight_raw, route_raw = parts
        # TODO: проверить, что courier_id существует
        if courier_id not in self.couriers:
            raise CourierNotFoundError(f"Курьер {courier_id} не найден в системе")
        # TODO: weight_raw преобразовать в float
        try:
            weight_kg = float(weight_raw)
        except ValueError as exc:
            raise WeightError(f"Некорректный формат веса: {weight_raw}") from exc
        # TODO: route_raw преобразовать в int
        try:
            route_min = int(route_raw)
        except ValueError as exc:
            raise RouteTimeError(f"Некорректный формат времени маршрута: {route_raw}") from exc
        # TODO: ошибки преобразования поднимать через WeightError / RouteTimeError с raise ... from exc
        # TODO: если weight_kg <= 0 -> raise WeightError(...)
        if weight_kg <= 0:
            raise WeightError(f"Вес должен быть положительным {weight_kg} кг")
        # TODO: если route_min <= 0 -> raise RouteTimeError(...)
        if route_min <= 0:
            raise RouteTimeError(f"Время маршрута должно быть положительным {route_min} мин")
        # TODO: вернуть Delivery(...)
        return Delivery(route_min, delivery_id, courier_id, client, weight_kg)

    def submit(self, row):
        # TODO: внутри try вызвать parse_delivery(row)
        try:
            delivery = self.parse_delivery(row)
            # TODO: если delivery.delivery_id уже в processed_ids -> raise DuplicateDeliveryError(...)
            if delivery.delivery_id in self.processed_ids:
                raise DuplicateDeliveryError(f"Дублируются ID доставки: {delivery.delivery_id}")
            # TODO: передать delivery в couriers[delivery.courier_id].assign(delivery)
            self.couriers[delivery.courier_id].assign(delivery)
            # TODO: после успеха обновить processed_ids и accepted
            self.processed_ids.add(delivery.delivery_id)
            self.accepted.append(delivery)
        except DeliveryError as e:
            # TODO: DeliveryError сохранить в errors как (row, error_type, message)
            error_type = type(e).__name__
            self.errors.append((row, error_type, str(e)))

    def load(self, rows):
        # TODO: вызвать submit(row) для каждой строки
        for row in rows:
            self.submit(row)

    def client_weights(self):
        # TODO: собрать dict вида client -> total_weight_kg
        weights: Dict[str, float] = {}
        for delivery in self.accepted:
            weights[delivery.client] = weights.get(delivery.client, 0) + delivery.weight_kg
        return weights

    def top_client(self):
        # TODO: использовать client_weights()
        client_weights = self.client_weights()
        # TODO: вернуть tuple(client, weight_kg) с максимумом
        if not client_weights:
            return None
        return max(client_weights.items(), key=lambda x: x[1])

    def busiest_courier(self):
        # TODO: найти курьера с максимумом total_route_time()
        busiest = None
        max_time = -1
        for courier_id, courier in self.couriers.items():
            total_time = courier.total_route_time()
            if total_time > max_time:
                max_time = total_time
                busiest = (courier_id, total_time)
        # TODO: вернуть tuple(courier_id, total_route_time)
        return busiest

    def low_charge_couriers(self, threshold=15):
        # TODO: вернуть список tuple(courier_id, charge_left)
        low_charge = []
        # TODO: включать только курьеров, у которых charge_left() <= threshold
        for courier_id, courier in self.couriers.items():
            charge_left = courier.charge_left()
            if charge_left <= threshold:
                low_charge.append((courier_id, charge_left))
        return low_charge

    def find_delivery(self, delivery_id):
        # TODO: пройтись по всем курьерам и их доставкам
        for courier in self.couriers.values():
            # TODO: если delivery.delivery_id совпал -> вернуть объект Delivery
            for delivery in courier.deliveries:
                if delivery.delivery_id == delivery_id:
                    return delivery
        # TODO: если доставка не найдена -> вернуть None
        return None


# TODO: создать сервис
service = CourierDispatchService(couriers)

# TODO: загрузить rows через service.load(rows)
service.load(rows)

# TODO: вывести принятые доставки
print("Принятые доставки:")
for delivery in service.accepted:
    print(f"  {delivery}")

# TODO: вывести ошибки
print("\nОшибки:")
for row, error_type, message in service.errors:
    print(f"  Строка: {row} | Ошибка: {error_type} | Сообщение: {message}")

# TODO: вывести по каждому курьеру deliveries, total_route_time и charge_left
print("\nИнформация по курьерам:")
for courier_id, courier in service.couriers.items():
    print(f"Курьер {courier_id}:")
    print(f"  Доставки: {courier.deliveries}")
    print(f"  Общее время маршрутов: {courier.total_route_time()} мин")
    print(f"  Остаток заряда: {courier.charge_left()} мин")

# TODO: вывести top_client()
top_client = service.top_client()
print(f"\nТоп‑клиент: {top_client}")

# TODO: вывести busiest_courier()
busiest = service.busiest_courier()
print(f"Самый загруженный курьер: {busiest}")

# TODO: вывести low_charge_couriers()
low_charge = service.low_charge_couriers()
print(f"Курьеры с низким зарядом (≤ 15 мин): {low_charge}")

# TODO: вывести find_delivery('DL-105')
delivery_105 = service.find_delivery('DL-105')
print(f"Доставка DL-105: {delivery_105}")

        
